In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ========== 1. 数据准备 ==========
weights = models.ResNet18_Weights.DEFAULT
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder("data/train", transform=train_transform)
val_dataset = datasets.ImageFolder("data/val", transform=val_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

# ========== 2. 加载预训练模型并修改 ==========
model = models.resnet18(weights=weights)
for param in model.parameters():
    param.requires_grad = False
num_classes = len(train_dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

# 解冻 layer4，和分类头一起训练
for param in model.layer4.parameters():
    param.requires_grad = True

# ========== 3. 训练配置 ==========
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)
criterion = nn.CrossEntropyLoss()
# 不同层用不同学习率
optimizer = torch.optim.Adam([
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
])
epochs = 10

# ========== 4. 训练循环 ==========
for epoch in range(epochs):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    avg_loss = total_loss / total

    # 验证
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2%} | Val Acc: {val_acc:.2%}")

# 保存模型
torch.save(model.state_dict(), "flower_model_2.pth")
print("模型已保存")

Epoch 1/10 | Loss: 0.7301 | Train Acc: 72.00% | Val Acc: 86.67%
Epoch 2/10 | Loss: 0.2879 | Train Acc: 91.00% | Val Acc: 96.67%
Epoch 3/10 | Loss: 0.2503 | Train Acc: 90.33% | Val Acc: 95.00%
Epoch 4/10 | Loss: 0.1076 | Train Acc: 96.33% | Val Acc: 95.00%
Epoch 5/10 | Loss: 0.1172 | Train Acc: 97.33% | Val Acc: 96.67%
Epoch 6/10 | Loss: 0.1115 | Train Acc: 96.00% | Val Acc: 100.00%
Epoch 7/10 | Loss: 0.1008 | Train Acc: 96.00% | Val Acc: 100.00%
Epoch 8/10 | Loss: 0.0888 | Train Acc: 96.33% | Val Acc: 98.33%
Epoch 9/10 | Loss: 0.1172 | Train Acc: 97.33% | Val Acc: 98.33%
Epoch 10/10 | Loss: 0.0841 | Train Acc: 96.33% | Val Acc: 100.00%
模型已保存
